In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import PchipInterpolator

# ============================================================
# Files
# ============================================================

CALIBRATION_FILE = "condensate_FR_calibration.csv"
UNKNOWN_FILE = "unknown_condensate_FR.csv"

TOTAL_ADENYLATE = 500.0  # uM, if assuming [ATP] + [ADP] = 500 uM


# ============================================================
# Load calibration
# ============================================================

cal = pd.read_csv(CALIBRATION_FILE)

required_cal_cols = {"ATP_uM", "ADP_uM", "FR_mean", "FR_sd"}
missing = required_cal_cols - set(cal.columns)
if missing:
    raise ValueError(f"Calibration file is missing columns: {missing}")

cal = cal.sort_values("ATP_uM").reset_index(drop=True)

# Check that FR decreases as ATP increases
if not np.all(np.diff(cal["FR_mean"]) < 0):
    print("Warning: FR is not strictly decreasing with ATP.")
    print("Check calibration data or consider using a fitted model.")

# Normalize FR only for visualization
FR_ADP_ONLY = cal.loc[cal["ATP_uM"].idxmin(), "FR_mean"]
FR_HIGH_ATP = cal.loc[cal["ATP_uM"].idxmax(), "FR_mean"]

cal["FR_norm"] = (cal["FR_mean"] - FR_HIGH_ATP) / (FR_ADP_ONLY - FR_HIGH_ATP)

# ============================================================
# Build inverse calibration: FR -> ATP
# ============================================================

# PCHIP needs x-values increasing.
# Since FR decreases with ATP, sort by FR.
cal_inv = cal.sort_values("FR_mean").reset_index(drop=True)

FR_values = cal_inv["FR_mean"].to_numpy()
ATP_values = cal_inv["ATP_uM"].to_numpy()

FR_min = FR_values.min()
FR_max = FR_values.max()
ATP_min = cal["ATP_uM"].min()
ATP_max = cal["ATP_uM"].max()

# Smooth monotonic interpolation
ATP_from_FR = PchipInterpolator(FR_values, ATP_values, extrapolate=False)

# Also make forward curve for plotting
ATP_grid = np.linspace(ATP_min, ATP_max, 500)
FR_from_ATP = PchipInterpolator(
    cal["ATP_uM"].to_numpy(),
    cal["FR_mean"].to_numpy(),
    extrapolate=False
)
FR_grid = FR_from_ATP(ATP_grid)


# ============================================================
# Function to estimate ATP from FR
# ============================================================

def estimate_ATP_from_FR(FR_mean, FR_sd=None):
    """
    Convert condensate ratiometric fluorescence FR = Cy5 / Atto488
    into estimated ATP concentration using experimental calibration.

    Returns:
    ATP_est, ATP_sd, FR_used, outside_calibration_range
    """

    FR_mean = np.asarray(FR_mean, dtype=float)

    outside = (FR_mean < FR_min) | (FR_mean > FR_max)
    FR_used = np.clip(FR_mean, FR_min, FR_max)

    ATP_est = ATP_from_FR(FR_used)

    if FR_sd is None:
        ATP_sd = np.full_like(ATP_est, np.nan, dtype=float)
    else:
        FR_sd = np.asarray(FR_sd, dtype=float)

        # Propagate uncertainty using local derivative dATP/dFR
        dATP_dFR = ATP_from_FR.derivative()(FR_used)
        ATP_sd = np.abs(dATP_dFR) * FR_sd

    return ATP_est, ATP_sd, FR_used, outside


# ============================================================
# Plot calibration curve
# ============================================================

plt.figure(figsize=(6, 4))

plt.errorbar(
    cal["ATP_uM"],
    cal["FR_mean"],
    yerr=cal["FR_sd"],
    fmt="o",
    capsize=3,
    label="Experimental calibration"
)

plt.plot(
    ATP_grid,
    FR_grid,
    linewidth=2,
    label="PCHIP interpolation"
)

plt.xlabel("[ATP] in condensate calibration (uM)")
plt.ylabel("FR = Cy5 / Atto488")
plt.title("Condensate ATP calibration curve")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig("condensate_FR_calibration_curve.png", dpi=300)
plt.show()


# ============================================================
# Plot normalized calibration curve
# ============================================================

FR_norm_grid = (FR_grid - FR_HIGH_ATP) / (FR_ADP_ONLY - FR_HIGH_ATP)

plt.figure(figsize=(6, 4))

plt.errorbar(
    cal["ATP_uM"],
    cal["FR_norm"],
    yerr=cal["FR_sd"] / (FR_ADP_ONLY - FR_HIGH_ATP),
    fmt="o",
    capsize=3,
    label="Normalized calibration"
)

plt.plot(
    ATP_grid,
    FR_norm_grid,
    linewidth=2,
    label="Interpolation"
)

plt.xlabel("[ATP] (uM)")
plt.ylabel("Normalized FR")
plt.title("Normalized condensate ATP calibration")
plt.ylim(-0.05, 1.05)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig("condensate_FR_calibration_normalized.png", dpi=300)
plt.show()


# ============================================================
# Save calibration table
# ============================================================

cal.to_csv("processed_condensate_FR_calibration.csv", index=False)


# ============================================================
# Estimate ATP in unknown samples
# ============================================================

unknown = pd.read_csv(UNKNOWN_FILE)

if "FR_mean" not in unknown.columns:
    raise ValueError("Unknown file must contain FR_mean column.")

if "FR_sd" not in unknown.columns:
    unknown["FR_sd"] = np.nan

unknown["ATP_est_uM"], unknown["ATP_sd_uM"], unknown["FR_used"], unknown["outside_calibration_range"] = estimate_ATP_from_FR(
    unknown["FR_mean"],
    unknown["FR_sd"]
)

# If assuming ATP + ADP = 500 uM
unknown["ADP_est_uM"] = TOTAL_ADENYLATE - unknown["ATP_est_uM"]
unknown["ADP_sd_uM"] = unknown["ATP_sd_uM"]

# SEM, if n is provided
if "n" in unknown.columns:
    unknown["ATP_sem_uM"] = unknown["ATP_sd_uM"] / np.sqrt(unknown["n"])
    unknown["ADP_sem_uM"] = unknown["ADP_sd_uM"] / np.sqrt(unknown["n"])

unknown.to_csv("estimated_condensate_ATP_from_FR.csv", index=False)

print(unknown)


# ============================================================
# Optional time-course plot, if time_min exists
# ============================================================

if "time_min" in unknown.columns:
    if "condition" not in unknown.columns:
        unknown["condition"] = "sample"

    for condition, sub in unknown.groupby("condition"):
        sub = sub.sort_values("time_min")

        plt.figure(figsize=(6, 4))

        plt.errorbar(
            sub["time_min"],
            sub["ATP_est_uM"],
            yerr=sub["ATP_sd_uM"],
            fmt="o-",
            capsize=3,
            label="[ATP]"
        )

        plt.errorbar(
            sub["time_min"],
            sub["ADP_est_uM"],
            yerr=sub["ADP_sd_uM"],
            fmt="o-",
            capsize=3,
            label="[ADP]"
        )

        plt.xlabel("Time (min)")
        plt.ylabel("Estimated concentration (uM)")
        plt.title(f"Estimated condensate ATP/ADP: {condition}")
        plt.ylim(-20, TOTAL_ADENYLATE + 20)
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()

        safe_name = str(condition).replace(" ", "_").replace("/", "_")
        plt.savefig(f"estimated_condensate_ATP_ADP_{safe_name}.png", dpi=300)
        plt.show()